In [ ]:
from spark_utils import SparkUtils
su = SparkUtils()
su

In [24]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [25]:
from pyspark.sql.types import TimestampType, FloatType
from datetime import datetime

factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_data_schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("temperature", FloatType(), True)
])


factorydf = su._spark.createDataFrame(factory_data, schema=factory_data_schema)
factorydf.show()

+----------+-------------------+-----------+
|machine_id|          timestamp|temperature|
+----------+-------------------+-----------+
|      M001|2026-01-26 08:00:00|       75.3|
|      M002|2026-01-26 08:05:00|       68.7|
|      M001|2026-01-26 08:10:00|       76.1|
|      M003|2026-01-26 08:15:00|       72.4|
|      M002|2026-01-26 08:20:00|       69.8|
|      M001|2026-01-26 08:25:00|       77.5|
|      M003|2026-01-26 08:30:00|       73.2|
|      M002|2026-01-26 08:35:00|       70.1|
|      M001|2026-01-26 08:40:00|       78.0|
|      M003|2026-01-26 08:45:00|       74.6|
+----------+-------------------+-----------+



## Exploring the schema and showing machines with temperature > 75

In [26]:
from pyspark.sql.functions import col

factorydf.printSchema()


df_filtered = factorydf.filter(col("temperature") > 75)
df_filtered.show()

root
 |-- machine_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: float (nullable = true)



+----------+-------------------+-----------+
|machine_id|          timestamp|temperature|
+----------+-------------------+-----------+
|      M001|2026-01-26 08:00:00|       75.3|
|      M001|2026-01-26 08:10:00|       76.1|
|      M001|2026-01-26 08:25:00|       77.5|
|      M001|2026-01-26 08:40:00|       78.0|
+----------+-------------------+-----------+



## Average temperature grouped by machine

In [27]:
average_temp = factorydf.groupBy("machine_id").avg("temperature")
average_temp.show()

+----------+-----------------+
|machine_id| avg(temperature)|
+----------+-----------------+
|      M002|69.53333282470703|
|      M003|73.39999898274739|
|      M001|76.72500038146973|
+----------+-----------------+



## minimum and maximum temperature by machine

In [28]:
from pyspark.sql.functions import max, min
max_min = factorydf.groupBy("machine_id").agg(max("temperature"),min("temperature"))
max_min.show()

+----------+----------------+----------------+
|machine_id|max(temperature)|min(temperature)|
+----------+----------------+----------------+
|      M002|            70.1|            68.7|
|      M001|            78.0|            75.3|
|      M003|            74.6|            72.4|
+----------+----------------+----------------+



## Count entries by machine

In [29]:
number = factorydf.groupBy("machine_id").count()
number.show()

+----------+-----+
|machine_id|count|
+----------+-----+
|      M002|    3|
|      M001|    4|
|      M003|    3|
+----------+-----+



## Show the hotest temperature registered

In [30]:
hotest=factorydf.orderBy(col("temperature").desc()).limit(1).show()
hotest

+----------+-------------------+-----------+
|machine_id|          timestamp|temperature|
+----------+-------------------+-----------+
|      M001|2026-01-26 08:40:00|       78.0|
+----------+-------------------+-----------+



In [34]:
#close the spark session
su._spark.stop()